# k03 — Tuning (STAGE2_DESIGN_FROZEN_v1.1 §8, C1–C3)
All four sets, train_01 only (test cells never extracted). 2 recording-level folds; Rule, LightGBM, LightGBM+S, DeepSets, GRU, GraphSAGE (8 configs each) + GraphSAGE-rewired with GraphSAGE's selection. Early stopping patience 3, cap 20; v1.1 tie-break. Saves selection.json and out-of-fold validation scores per set. Validation numbers are selection statistics, not results.

In [ ]:
import os
os.makedirs('/kaggle/working/code', exist_ok=True)
FILES = {'feats.py': 'import numpy as np, pandas as pd, re, os, time\nW = 64\nHEXV = np.full(256, 0, dtype=np.uint8)\nfor i, ch in enumerate(\'0123456789abcdef\'):\n    HEXV[ord(ch)] = i; HEXV[ord(ch.upper())] = i\nPOP = np.array([bin(i).count(\'1\') for i in range(256)], dtype=np.uint8)\n\ndef slog(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\ndef load_file(path):\n    df = pd.read_csv(path, dtype={\'arbitration_id\': str, \'data_field\': str, \'attack\': np.int8}, keep_default_na=True)\n    ts = df[\'timestamp\'].to_numpy(np.float64)\n    ids = df[\'arbitration_id\'].str.rjust(3, \'0\').str[-3:]\n    ib = np.frombuffer(\'\'.join(ids.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 3)\n    idv = HEXV[ib].astype(np.int32)\n    id_int = idv[:, 0] * 256 + idv[:, 1] * 16 + idv[:, 2]\n    d = df[\'data_field\'].fillna(\'\')\n    plen = (d.str.len().to_numpy() // 2).clip(0, 8).astype(np.int8)\n    d16 = d.str[:16].str.ljust(16, \'0\')\n    db = np.frombuffer(\'\'.join(d16.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 16)\n    nib = HEXV[db]\n    pay = (nib[:, 0::2] * 16 + nib[:, 1::2]).astype(np.uint8)\n    posmask = np.arange(8)[None, :] < plen[:, None]\n    pay = np.where(posmask, pay, 0).astype(np.uint8)\n    y = df[\'attack\'].to_numpy(np.int8)\n    return ts, id_int, plen, pay, y\n\ndef per_frame_globals(ts, id_int, plen, pay):\n    n = len(ts); idx = np.arange(n)\n    order = np.lexsort((idx, id_int))\n    prev = np.full(n, -1, dtype=np.int64)\n    same = np.r_[False, id_int[order][1:] == id_int[order][:-1]]\n    prev[order[same]] = order[np.flatnonzero(same) - 1]\n    has = prev >= 0\n    pp = np.where(has, prev, 0)\n    dt_same = np.where(has, ts - ts[pp], 0.0)\n    x = pay ^ pay[pp]\n    ham = np.where(has, POP[x].sum(1), 0).astype(np.float32)\n    maxlen = np.maximum(plen, plen[pp]).astype(np.float32)\n    chg = np.where(has, (x != 0).sum(1) / np.maximum(maxlen, 1), 0).astype(np.float32)\n    lenchg = np.where(has, plen != plen[pp], False)\n    ent = np.zeros(n, dtype=np.float32)\n    for s in range(0, n, 500000):\n        b = pay[s:s + 500000]; L = plen[s:s + 500000].astype(np.int32)\n        valid = np.arange(8)[None, :] < L[:, None]\n        eq = (b[:, :, None] == b[:, None, :]) & valid[:, :, None] & valid[:, None, :]\n        c = eq.sum(2).astype(np.float32)\n        Lf = np.maximum(L, 1).astype(np.float32)[:, None]\n        with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n            term = np.where(valid, np.log2(np.where(c > 0, c, 1) / Lf), 0.0)\n        ent[s:s + 500000] = -(term.sum(1) / Lf[:, 0])\n    return prev, dt_same, ham, chg, ent, lenchg\n\nFRAME_NAMES = [\'plen\', \'dt_prev_any\', \'dt_same\', \'no_prev_same_in_window\', \'hamming\', \'changed_frac\', \'entropy\', \'same_as_prev\']\nNODE_NAMES = [\'count\', \'first_pos\', \'last_pos\', \'ia_mean\', \'ia_min\', \'ia_max\', \'ia_missing\', \'plen_mean\', \'plen_max\', \'plen_changes\', \'ham_mean\', \'chg_mean\', \'ent_mean\']\nGLOBAL_NAMES = [\'duration\', \'distinct_ids\', \'distinct_transitions\', \'fps\']\n\ndef windows_for_file(path, stride, id_perm=None):\n    ts, id_int, plen, pay, y = load_file(path)\n    if id_perm is not None:\n        id_int = id_perm[id_int]\n    n = len(ts)\n    if n < W:\n        return None\n    prev, dt_same_g, ham_g, chg_g, ent_g, lenchg_g = per_frame_globals(ts, id_int, plen, pay)\n    starts = np.arange(0, n - W + 1, stride)\n    nw = len(starts)\n    I = starts[:, None] + np.arange(W)[None, :]\n    ok = prev[I] >= starts[:, None]\n    tsw = ts[I]\n    dtp = np.diff(tsw, axis=1, prepend=tsw[:, :1])\n    idw = id_int[I]\n    fr = np.zeros((nw, W, len(FRAME_NAMES)), dtype=np.float32)\n    fr[..., 0] = plen[I] / 8.0\n    fr[..., 1] = slog(dtp * 1000)\n    fr[..., 2] = np.where(ok, slog(dt_same_g[I] * 1000), 0)\n    fr[..., 3] = ~ok\n    fr[..., 4] = np.where(ok, ham_g[I] / 64.0, 0)\n    fr[..., 5] = np.where(ok, chg_g[I], 0)\n    fr[..., 6] = ent_g[I] / 3.0\n    fr[:, 1:, 7] = idw[:, 1:] == idw[:, :-1]\n    # nodes\n    key = (np.arange(nw)[:, None] * 4096 + idw).ravel()\n    uk, inv = np.unique(key, return_inverse=True)\n    inv = inv.reshape(nw, W)\n    win_of_node = uk // 4096\n    node_first = np.searchsorted(win_of_node, np.arange(nw))\n    local = inv - node_first[:, None]\n    nn = np.bincount(win_of_node, minlength=nw)\n    G = len(uk)\n    fl = inv.ravel()\n    pos = np.broadcast_to(np.arange(W), (nw, W)).ravel()\n    okf = ok.ravel()\n    def agg_sum(v, m=None):\n        return np.bincount(fl, weights=(v if m is None else v * m), minlength=G)\n    order = np.argsort(fl, kind=\'stable\'); fs = fl[order]\n    bnd = np.flatnonzero(np.r_[True, fs[1:] != fs[:-1]])\n    def agg_min(v): return np.minimum.reduceat(v[order], bnd)\n    def agg_max(v): return np.maximum.reduceat(v[order], bnd)\n    cnt = np.bincount(fl, minlength=G).astype(np.float32)\n    iak = np.where(okf, slog(dt_same_g[I].ravel() * 1000), np.nan)\n    nia = agg_sum(okf.astype(np.float64))\n    has_ia = nia > 0\n    ia_mean = np.where(has_ia, agg_sum(np.nan_to_num(iak)) / np.maximum(nia, 1), 0)\n    ia_min = np.where(has_ia, agg_min(np.where(okf, iak, np.inf)), 0)\n    ia_max = np.where(has_ia, agg_max(np.where(okf, iak, -np.inf)), 0)\n    pl = (plen[I].ravel()).astype(np.float64)\n    nd = np.zeros((G, len(NODE_NAMES)), dtype=np.float32)\n    nd[:, 0] = cnt / W\n    nd[:, 1] = agg_min(pos.astype(np.float64)) / W\n    nd[:, 2] = agg_max(pos.astype(np.float64)) / W\n    nd[:, 3] = ia_mean; nd[:, 4] = ia_min; nd[:, 5] = ia_max\n    nd[:, 6] = ~has_ia\n    nd[:, 7] = agg_sum(pl) / cnt / 8.0\n    nd[:, 8] = agg_max(pl) / 8.0\n    nd[:, 9] = agg_max(pl) != agg_min(pl)\n    nd[:, 10] = np.where(has_ia, agg_sum(ham_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1) / 64.0, 0)\n    nd[:, 11] = np.where(has_ia, agg_sum(chg_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1), 0)\n    nd[:, 12] = agg_sum(ent_g[I].ravel().astype(np.float64)) / cnt / 3.0\n    node = np.zeros((nw, W, len(NODE_NAMES)), dtype=np.float32)\n    node[win_of_node, np.arange(G) - node_first[win_of_node]] = nd\n    # edges: local src/dst per transition\n    src = local[:, :-1].astype(np.uint8); dst = local[:, 1:].astype(np.uint8)\n    tr = np.unique((np.arange(nw)[:, None] * 4096 + local[:, :-1] * 64 + local[:, 1:]).ravel())\n    ntr = np.bincount(tr // 4096, minlength=nw)\n    dur = tsw[:, -1] - tsw[:, 0]\n    glob = np.stack([slog(dur * 1000), nn / W, ntr / 63.0, slog(W / np.maximum(dur, 1e-6))], 1).astype(np.float32)\n    lab = (y[I].max(1) > 0).astype(np.int8)\n    nattack = y[I].sum(1).astype(np.int16)\n    return dict(frame=fr.astype(np.float16), node=node.astype(np.float16), nmask=(np.arange(W)[None, :] < nn[:, None]),\n                src=src, dst=dst, glob=glob, y=lab, nattack=nattack, starts=starts.astype(np.int64), t0=tsw[:, 0], t1=tsw[:, -1])\n\nSTRUCT_NAMES = [\'transition_entropy\', \'unique_transition_ratio\', \'self_loop_ratio\', \'mean_out_degree\',\n                \'max_out_degree\', \'max_in_degree\', \'degree_entropy\', \'density\']\n\ndef structural_features(src, dst, nmask):\n    """Explicit structural/topological summaries of each window\'s directed transition multigraph.\n    src, dst: (nw, 63) local node indices of consecutive frames; nmask: (nw, 64) valid nodes.\n    Uses only ID-free graph structure (local node indices are arbitrary labels)."""\n    nw, E = src.shape\n    n = nmask.sum(1).astype(np.float64)\n    s = src.astype(np.int64); d = dst.astype(np.int64)\n    w = np.repeat(np.arange(nw), E)\n    key = w * 4096 + (s * 64 + d).ravel()\n    uk, cnt = np.unique(key, return_counts=True)\n    uw = uk // 4096; us = (uk % 4096) // 64; ud = (uk % 4096) % 64\n    p = cnt / float(E)\n    ent = np.bincount(uw, weights=-p * np.log2(p), minlength=nw) / np.log2(E)\n    uniq = np.bincount(uw, minlength=nw) / float(E)\n    selfr = (s == d).sum(1) / float(E)\n    ns = us != ud\n    outdeg = np.bincount(uw[ns] * 64 + us[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    indeg = np.bincount(uw[ns] * 64 + ud[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    e_ns = np.bincount(uw[ns], minlength=nw).astype(np.float64)\n    mean_out = np.where(n > 0, e_ns / np.maximum(n, 1), 0)\n    tot = outdeg + indeg; ts = tot.sum(1, keepdims=True)\n    pd_ = np.where(ts > 0, tot / np.maximum(ts, 1), 0)\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        h = -(np.where(pd_ > 0, pd_ * np.log2(np.where(pd_ > 0, pd_, 1)), 0)).sum(1)\n    deg_ent = np.where(n > 1, h / np.log2(np.maximum(n, 2)), 0)\n    dens = np.where(n > 1, e_ns / np.maximum(n * (n - 1), 1), 0)\n    return np.stack([ent, uniq, selfr, mean_out, outdeg.max(1), indeg.max(1), deg_ent, dens], 1).astype(np.float32)\n', 'models.py': "import torch, torch.nn as nn, numpy as np, time\nW = 64\n\ndef mlp(i, h, o, drop):\n    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o))\n\nclass Head(nn.Module):\n    def __init__(self, h, g, drop):\n        super().__init__(); self.rho = mlp(3 * h + g, h, 1, drop)\n    def forward(self, H, mask, glob):\n        m = mask.unsqueeze(-1).float()\n        s = (H * m).sum(1); mean = s / m.sum(1).clamp(min=1)\n        mx = H.masked_fill(m == 0, -1e4).max(1).values\n        return self.rho(torch.cat([s, mean, mx, glob], 1)).squeeze(-1)\n\nclass DeepSets(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, h), nn.ReLU())\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        return self.head(self.phi(b['node']), b['nmask'], b['glob'])\n\nclass SAGELayer(nn.Module):\n    def __init__(self, i, o):\n        super().__init__(); self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n    def forward(self, H, A):\n        # A[b, src, dst] = weight; aggregate incoming neighbours of each dst node (weighted mean)\n        agg = torch.bmm(A.transpose(1, 2), H)\n        deg = A.sum(1).unsqueeze(-1)\n        agg = agg / deg.clamp(min=1e-9)\n        return self.self_lin(H) + self.nei_lin(agg)\n\nclass GraphSAGE(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.l1 = SAGELayer(f, h); self.l2 = SAGELayer(h, h); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b['adj']; m = b['nmask'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b['node'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b['nmask'], b['glob'])\n\nclass GRUNet(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.gru = nn.GRU(f, h, batch_first=True); self.out = mlp(2 * h + g, h, 1, drop)\n    def forward(self, b):\n        O, hn = self.gru(b['frame'])\n        return self.out(torch.cat([hn[-1], O.mean(1), b['glob']], 1)).squeeze(-1)\n\ndef nparams(m):\n    return sum(p.numel() for p in m.parameters())\n\ndef build_adj(src, dst, B, device):\n    A = torch.zeros(B, W, W, device=device)\n    bi = torch.arange(B, device=device).unsqueeze(1).expand_as(src)\n    A.index_put_((bi.reshape(-1), src.reshape(-1).long(), dst.reshape(-1).long()), torch.full((src.numel(),), 1.0 / 63, device=device), accumulate=True)\n    return A\n\ndef rewire_dst(dst, gen):\n    # degree-preserving directed rewiring: permute destination endpoints among a window's 63 edges\n    # (every source keeps its out-degree, every destination keeps its in-degree, multiplicities included)\n    r = torch.rand(dst.shape, generator=gen, device=dst.device)\n    perm = r.argsort(1)\n    return torch.gather(dst, 1, perm)\n\ndef edge_change_fraction(src, dst, dst2):\n    # fraction of the 63 directed edges (as a multiset per window) not present in the original\n    B = src.shape[0]\n    k1 = (src.long() * 64 + dst.long()).sort(1).values\n    k2 = (src.long() * 64 + dst2.long()).sort(1).values\n    fr = []\n    for i in range(B):\n        a, ca = torch.unique(k1[i], return_counts=True); b2, cb = torch.unique(k2[i], return_counts=True)\n        common = 0\n        d = dict(zip(a.tolist(), ca.tolist()))\n        for kk, cc in zip(b2.tolist(), cb.tolist()):\n            common += min(cc, d.get(kk, 0))\n        fr.append(1 - common / 63.0)\n    return float(np.mean(fr))\n", 'exp_tune.py': '"""Stage 3 tuning (STAGE2_DESIGN_FROZEN_v1.1 §8 + C1-C3). Reads ONLY train_01 of the requested sets.\nUsage: python exp_tune.py --sets set_01,set_03 --device cuda:0 --out /kaggle/working/tune"""\nimport os, sys, re, json, time, glob, hashlib, argparse, subprocess, traceback, zipfile\nimport numpy as np, torch\nfrom sklearn.metrics import average_precision_score, log_loss\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport feats, models\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--sets\', required=True); ap.add_argument(\'--device\', default=\'cuda:0\')\nap.add_argument(\'--out\', default=\'/kaggle/working/tune\'); ap.add_argument(\'--lgb_threads\', type=int, default=2)\nargs = ap.parse_args()\nDEV = args.device; OUT = args.out; os.makedirs(OUT, exist_ok=True)\nSETS = args.sets.split(\',\')\nEPOCH_CAP, PATIENCE, BS = 20, 3, 1024\nZIP = glob.glob(\'/kaggle/input/**/can-train-and-test-v1.zip\', recursive=True)[0]\nHASHES = glob.glob(\'/kaggle/input/**/file_hashes_sha256.csv\', recursive=True)[0]\nDATA = \'/kaggle/temp/data\'\nLOG = open(os.path.join(OUT, f\'log_{"_".join(SETS)}.txt\'), \'a\')\ndef log(*a):\n    s = time.strftime(\'%H:%M:%S \') + \' \'.join(str(x) for x in a); print(s, flush=True); LOG.write(s + \'\\n\'); LOG.flush()\n\ndef extract_train(st):\n    H = {}\n    for line in open(HASHES).read().strip().split(\'\\n\')[1:]:\n        rel, size, sha = line.split(\',\'); H[rel] = (int(size), sha)\n    with zipfile.ZipFile(ZIP) as z:\n        names = [n for n in z.namelist() if n.startswith(f\'can-train-and-test/{st}/train_01/\') and n.endswith(\'.csv\')]\n        assert names and all(\'/train_01/\' in n for n in names)\n        for n in names:\n            dest = os.path.join(DATA, n)\n            if not os.path.exists(dest):\n                z.extract(n, DATA)\n    files = sorted(glob.glob(os.path.join(DATA, \'can-train-and-test\', st, \'train_01\', \'*.csv\')))\n    for p in files:\n        rel = p.split(\'can-train-and-test/\')[1]; hh = hashlib.sha256()\n        with open(p, \'rb\') as f:\n            for b in iter(lambda: f.read(8 << 20), b\'\'): hh.update(b)\n        assert (os.path.getsize(p), hh.hexdigest()) == H[rel], \'hash mismatch \' + rel\n    return files\n\ndef fam(p): return re.sub(r\'-\\d+\\.csv$\', \'\', os.path.basename(p))\ndef idx(p): return int(re.search(r\'-(\\d+)\\.csv$\', p).group(1))\n\nKEYS = [\'frame\', \'node\', \'nmask\', \'src\', \'dst\', \'glob\', \'y\']\ndef build_features(files):\n    F = {}\n    for p in files:\n        d = feats.windows_for_file(p, 32)\n        d[\'struct\'] = feats.structural_features(d[\'src\'], d[\'dst\'], d[\'nmask\'])\n        F[p] = d\n    return F\n\ndef cat(F, plist, stride64):\n    out = {k: [] for k in KEYS + [\'struct\']}\n    for p in plist:\n        d = F[p]; sel = (d[\'starts\'] % 64 == 0) if stride64 else slice(None)\n        for k in out: out[k].append(d[k][sel])\n    return {k: np.concatenate(v) for k, v in out.items()}\n\ndef flat_features(D, with_struct):\n    m = D[\'nmask\'][..., None]; x = D[\'node\'].astype(np.float32); cnt = m.sum(1).clip(1)\n    mean = (x * m).sum(1) / cnt; std = np.sqrt(((x - mean[:, None]) ** 2 * m).sum(1) / cnt)\n    mn = np.where(m, x, np.inf).min(1); mx = np.where(m, x, -np.inf).max(1)\n    X = [mean, std, mn, mx, D[\'glob\']]\n    if with_struct: X.append(D[\'struct\'])\n    return np.concatenate(X, 1).astype(np.float32)\n\nNF, NN, NG = len(feats.FRAME_NAMES), len(feats.NODE_NAMES), len(feats.GLOBAL_NAMES)\ndef make(name, h, drop):\n    if name in (\'GraphSAGE\', \'GraphSAGE_rewired\'): return models.GraphSAGE(NN, h, NG, drop)\n    if name == \'DeepSets\': return models.DeepSets(NN, h, NG, drop)\n    if name == \'GRU\': return models.GRUNet(NF, h, NG, drop)\ndef matched_hidden(name, h_sage):\n    if name in (\'GraphSAGE\', \'GraphSAGE_rewired\'): return h_sage\n    target = models.nparams(make(\'GraphSAGE\', h_sage, 0.0))\n    return min(range(4, 513), key=lambda h: abs(models.nparams(make(name, h, 0.0)) - target))\n\nNEURAL_GRID = [dict(lr=lr, h_sage=h, dropout=dr) for lr in (1e-3, 3e-4) for h in (32, 64) for dr in (0.0, 0.2)]\nLGB_GRID = [dict(num_leaves=nl, learning_rate=lr, min_child_samples=mc) for nl in (31, 63) for lr in (0.05, 0.1) for mc in (20, 100)]\n\ndef to_gpu(D, norm):\n    T = {}\n    T[\'frame\'] = ((torch.from_numpy(D[\'frame\'].astype(np.float32)).to(DEV) - norm[\'fm\']) / norm[\'fs\']).half()\n    nm = torch.from_numpy(D[\'nmask\']).to(DEV)\n    T[\'node\'] = (((torch.from_numpy(D[\'node\'].astype(np.float32)).to(DEV) - norm[\'nm\']) / norm[\'ns\']) * nm.unsqueeze(-1)).half()\n    T[\'nmask\'] = nm\n    T[\'glob\'] = ((torch.from_numpy(D[\'glob\']).to(DEV) - norm[\'gm\']) / norm[\'gs\'])\n    T[\'src\'] = torch.from_numpy(D[\'src\']).to(DEV); T[\'dst\'] = torch.from_numpy(D[\'dst\']).to(DEV)\n    T[\'y\'] = torch.from_numpy(D[\'y\'].astype(np.float32)).to(DEV)\n    return T\n\ndef norm_stats(D):\n    f = D[\'frame\'].astype(np.float32).reshape(-1, NF); msk = D[\'nmask\'].reshape(-1)\n    n = D[\'node\'].astype(np.float32).reshape(-1, NN)[msk]\n    T = lambda a: torch.tensor(a, dtype=torch.float32, device=DEV)\n    return {\'fm\': T(f.mean(0)), \'fs\': T(f.std(0) + 1e-6), \'nm\': T(n.mean(0)), \'ns\': T(n.std(0) + 1e-6), \'gm\': T(D[\'glob\'].mean(0)), \'gs\': T(D[\'glob\'].std(0) + 1e-6)}\n\ndef batch(T, ix, rewire, gen):\n    b = {\'frame\': T[\'frame\'][ix].float(), \'node\': T[\'node\'][ix].float(), \'nmask\': T[\'nmask\'][ix], \'glob\': T[\'glob\'][ix]}\n    dst = T[\'dst\'][ix]\n    if rewire: dst = models.rewire_dst(dst, gen)\n    b[\'adj\'] = models.build_adj(T[\'src\'][ix], dst, len(ix), DEV)\n    return b\n\ndef score(m, T, rewire, seed=12345):\n    m.eval(); g = torch.Generator(device=DEV); g.manual_seed(seed); out = []\n    with torch.no_grad():\n        for s in range(0, len(T[\'y\']), 4096):\n            ix = torch.arange(s, min(s + 4096, len(T[\'y\'])), device=DEV)\n            out.append(m(batch(T, ix, rewire, g)).float())\n    return torch.cat(out).cpu().numpy()\n\ndef train_neural(name, cfg, Ttr, Tva, yva, seed=0):\n    torch.manual_seed(seed); np.random.seed(seed)\n    gen = torch.Generator(device=DEV); gen.manual_seed(seed)\n    h = matched_hidden(name, cfg[\'h_sage\']); m = make(name, h, cfg[\'dropout\']).to(DEV)\n    opt = torch.optim.AdamW(m.parameters(), lr=cfg[\'lr\'])\n    pos = float(Ttr[\'y\'].sum()); neg = len(Ttr[\'y\']) - pos\n    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1.0), device=DEV))\n    rew = name == \'GraphSAGE_rewired\'; hist = []; best = (-1, None, None, None); bad = 0\n    n = len(Ttr[\'y\'])\n    for ep in range(EPOCH_CAP):\n        m.train(); t = time.time()\n        perm = torch.randperm(n, device=DEV, generator=gen)\n        for s in range(0, n, BS):\n            ix = perm[s:s + BS]\n            loss = lossf(m(batch(Ttr, ix, rew, gen)), Ttr[\'y\'][ix]); opt.zero_grad(); loss.backward(); opt.step()\n        sc = score(m, Tva, rew)\n        apv = float(average_precision_score(yva, sc)) if yva.sum() > 0 else float(\'nan\')\n        pr = 1 / (1 + np.exp(-np.clip(sc, -30, 30)))\n        bce = float(log_loss(yva, pr, labels=[0, 1]))\n        hist.append({\'epoch\': ep + 1, \'val_ap\': round(apv, 6), \'val_bce\': round(bce, 6), \'s\': round(time.time() - t, 1)})\n        if apv > best[0]:\n            best = (apv, ep + 1, bce, sc.astype(np.float32)); bad = 0\n        else:\n            bad += 1\n            if bad >= PATIENCE: break\n    return {\'hidden\': h, \'params\': models.nparams(m), \'best_ap\': best[0], \'best_epoch\': best[1], \'best_bce\': best[2], \'history\': hist}, best[3]\n\ndef train_lgb(cfg, Xtr, ytr, Xva, yva):\n    import lightgbm as lgb\n    pos = ytr.sum(); neg = len(ytr) - pos\n    clf = lgb.LGBMClassifier(n_estimators=1000, scale_pos_weight=neg / max(pos, 1), n_jobs=args.lgb_threads, verbose=-1, random_state=0, metric=\'average_precision\', **cfg)\n    clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False)])\n    bi = clf.best_iteration_ or 1000\n    sc = clf.predict_proba(Xva, num_iteration=bi)[:, 1]\n    return {\'best_ap\': float(average_precision_score(yva, sc)), \'best_epoch\': int(bi), \'best_bce\': float(log_loss(yva, sc, labels=[0, 1]))}, sc.astype(np.float32)\n\ndef select(runs, grid, size_key):\n    # runs[ci][fold] -> result dict ; tie-break per v1.1 C3\n    rows = []\n    for ci, cfg in enumerate(grid):\n        r = [runs[ci][f] for f in (1, 2)]\n        if any(x is None for x in r): continue\n        rows.append(dict(ci=ci, ap=np.mean([x[\'best_ap\'] for x in r]), bce=np.mean([x[\'best_bce\'] for x in r]),\n                         ep=np.mean([x[\'best_epoch\'] for x in r]), size=cfg[size_key], lr=cfg.get(\'lr\', cfg.get(\'learning_rate\'))))\n    best_ap = max(r[\'ap\'] for r in rows)\n    tied = [r for r in rows if r[\'ap\'] >= best_ap - 0.001]\n    tied.sort(key=lambda r: (r[\'bce\'], r[\'ep\'], r[\'size\'], r[\'lr\']))\n    ch = tied[0]\n    return {\'config_index\': ch[\'ci\'], \'config\': grid[ch[\'ci\']], \'mean_val_ap\': ch[\'ap\'], \'mean_val_bce\': ch[\'bce\'],\n            \'final_epochs\': max(1, int(round(ch[\'ep\']))), \'n_tied_within_0.001\': len(tied), \'all\': rows}\n\ndef run_set(st):\n    sd = os.path.join(OUT, st); os.makedirs(sd, exist_ok=True)\n    if os.path.exists(os.path.join(sd, \'selection.json\')):\n        log(st, \'already done\'); return\n    t0 = time.time(); files = extract_train(st); log(st, \'train files\', len(files), \'extract+verify s\', round(time.time() - t0, 1))\n    lo = min(idx(p) for p in files)\n    folds = {1: ([p for p in files if idx(p) == lo], [p for p in files if idx(p) != lo]),\n             2: ([p for p in files if idx(p) != lo], [p for p in files if idx(p) == lo])}\n    t0 = time.time(); F = build_features(files); log(st, \'features s\', round(time.time() - t0, 1))\n    FD = {f: (cat(F, folds[f][0], False), cat(F, folds[f][1], True)) for f in (1, 2)}\n    meta = {\'set\': st, \'folds\': {f: {\'train\': [os.path.basename(p) for p in folds[f][0]], \'val\': [os.path.basename(p) for p in folds[f][1]]} for f in (1, 2)},\n            \'windows\': {f: {\'train\': int(len(FD[f][0][\'y\'])), \'train_pos\': int(FD[f][0][\'y\'].sum()), \'val\': int(len(FD[f][1][\'y\'])), \'val_pos\': int(FD[f][1][\'y\'].sum())} for f in (1, 2)}}\n    json.dump(meta, open(os.path.join(sd, \'meta.json\'), \'w\'), indent=1)\n    SEL = {}; OOF = {}\n    # Rule: best single base feature (56) by mean validation AP over folds\n    Xb = {f: (flat_features(FD[f][0], False), flat_features(FD[f][1], False)) for f in (1, 2)}\n    best = None\n    for j in range(Xb[1][1].shape[1]):\n        for sg in (1, -1):\n            a = np.mean([average_precision_score(FD[f][1][\'y\'], sg * Xb[f][1][:, j]) for f in (1, 2)])\n            if best is None or a > best[0]: best = (a, j, sg)\n    SEL[\'Rule\'] = {\'feature_index\': best[1], \'sign\': best[2], \'mean_val_ap\': best[0]}\n    OOF[\'Rule\'] = {f: (best[2] * Xb[f][1][:, best[1]]).astype(np.float32) for f in (1, 2)}\n    log(st, \'Rule\', SEL[\'Rule\'])\n    # LightGBM and LightGBM+S\n    for name, ws in ((\'LightGBM\', False), (\'LightGBM_S\', True)):\n        X = {f: (flat_features(FD[f][0], ws), flat_features(FD[f][1], ws)) for f in (1, 2)}\n        runs = {ci: {} for ci in range(len(LGB_GRID))}; sc_store = {ci: {} for ci in range(len(LGB_GRID))}\n        for ci, cfg in enumerate(LGB_GRID):\n            for f in (1, 2):\n                t = time.time()\n                try:\n                    r, sc = train_lgb(cfg, X[f][0], FD[f][0][\'y\'], X[f][1], FD[f][1][\'y\']); runs[ci][f] = r; sc_store[ci][f] = sc\n                except Exception as e:\n                    runs[ci][f] = None; log(st, name, ci, f, \'ERROR\', repr(e))\n                log(st, name, \'cfg\', ci, \'fold\', f, runs[ci][f] and {k: runs[ci][f][k] for k in (\'best_ap\', \'best_epoch\')}, round(time.time() - t, 1), \'s\')\n        SEL[name] = select(runs, LGB_GRID, \'num_leaves\'); SEL[name][\'n_features\'] = int(X[1][0].shape[1])\n        OOF[name] = sc_store[SEL[name][\'config_index\']]\n        json.dump({\'runs\': runs}, open(os.path.join(sd, f\'runs_{name}.json\'), \'w\'), default=str)\n        log(st, name, \'selected\', SEL[name][\'config\'], SEL[name][\'mean_val_ap\'])\n    # Neural models\n    for f in (1, 2):\n        norm = norm_stats(FD[f][0])\n        FD[f] = (to_gpu(FD[f][0], norm), to_gpu(FD[f][1], norm), FD[f][1][\'y\'])\n    for name in (\'DeepSets\', \'GRU\', \'GraphSAGE\'):\n        runs = {ci: {} for ci in range(len(NEURAL_GRID))}; sc_store = {ci: {} for ci in range(len(NEURAL_GRID))}\n        for ci, cfg in enumerate(NEURAL_GRID):\n            for f in (1, 2):\n                t = time.time()\n                try:\n                    r, sc = train_neural(name, cfg, FD[f][0], FD[f][1], FD[f][2]); runs[ci][f] = r; sc_store[ci][f] = sc\n                except Exception as e:\n                    runs[ci][f] = None; log(st, name, ci, f, \'ERROR\', repr(e), traceback.format_exc()[-800:])\n                log(st, name, \'cfg\', ci, \'fold\', f, runs[ci][f] and {k: runs[ci][f][k] for k in (\'best_ap\', \'best_epoch\', \'params\')}, round(time.time() - t, 1), \'s\')\n        SEL[name] = select(runs, NEURAL_GRID, \'h_sage\')\n        SEL[name][\'hidden\'] = matched_hidden(name, SEL[name][\'config\'][\'h_sage\'])\n        OOF[name] = sc_store[SEL[name][\'config_index\']]\n        json.dump({\'runs\': runs}, open(os.path.join(sd, f\'runs_{name}.json\'), \'w\'), default=str)\n        log(st, name, \'selected\', SEL[name][\'config\'], SEL[name][\'mean_val_ap\'], \'epochs\', SEL[name][\'final_epochs\'])\n    # GraphSAGE_rewired: GraphSAGE\'s selected configuration, both folds (for out-of-fold scores), no selection\n    cfg = SEL[\'GraphSAGE\'][\'config\']; runs = {}; OOF[\'GraphSAGE_rewired\'] = {}\n    for f in (1, 2):\n        try:\n            r, sc = train_neural(\'GraphSAGE_rewired\', cfg, FD[f][0], FD[f][1], FD[f][2]); runs[f] = r; OOF[\'GraphSAGE_rewired\'][f] = sc\n        except Exception as e:\n            runs[f] = {\'best_ap\': None, \'error\': repr(e)}; OOF[\'GraphSAGE_rewired\'][f] = np.zeros(len(FD[f][2]), np.float32)\n            log(st, \'GraphSAGE_rewired\', f, \'ERROR\', repr(e), traceback.format_exc()[-800:])\n        log(st, \'GraphSAGE_rewired fold\', f, runs[f].get(\'best_ap\'))\n    SEL[\'GraphSAGE_rewired\'] = {\'config\': cfg, \'hidden\': SEL[\'GraphSAGE\'][\'hidden\'], \'final_epochs\': SEL[\'GraphSAGE\'][\'final_epochs\'],\n                                \'fold_best_ap\': {f: runs[f][\'best_ap\'] for f in (1, 2)}, \'note\': \'uses GraphSAGE selection (v1.0 §7)\'}\n    json.dump({\'runs\': runs}, open(os.path.join(sd, \'runs_GraphSAGE_rewired.json\'), \'w\'), default=str)\n    np.savez_compressed(os.path.join(sd, \'oof_scores.npz\'), **{f\'{k}_f{f}\': v[f] for k, v in OOF.items() for f in (1, 2)},\n                        **{f\'y_f{f}\': (FD[f][2]).astype(np.int8) for f in (1, 2)})\n    json.dump(SEL, open(os.path.join(sd, \'selection.json\'), \'w\'), indent=1, default=str)\n    log(st, \'DONE\')\n\nfor st in SETS:\n    try:\n        run_set(st)\n    except Exception as e:\n        log(st, \'FATAL\', repr(e), traceback.format_exc()[-2000:])\n'}
for n, s in FILES.items():
    open('/kaggle/working/code/' + n, 'w').write(s)
os.makedirs('/kaggle/temp', exist_ok=True)
import glob
print(sorted(os.listdir('/kaggle/working/code')), glob.glob('/kaggle/input/**/can-train-and-test-v1.zip', recursive=True))

In [ ]:
import subprocess, sys, time, shutil, glob, os
# resume support: copy finished set results from an attached previous version of this notebook, if any
for sel in glob.glob('/kaggle/input/**/tune/set_0*/selection.json', recursive=True):
    src = os.path.dirname(sel); dst = '/kaggle/working/tune/' + os.path.basename(src)
    if not os.path.exists(dst): shutil.copytree(src, dst)
cmd = lambda sets, dev: [sys.executable, '/kaggle/working/code/exp_tune.py', '--sets', sets, '--device', dev, '--lgb_threads', '2']
t0 = time.time()
pA = subprocess.Popen(cmd('set_01,set_03', 'cuda:0'))
pB = subprocess.Popen(cmd('set_02,set_04', 'cuda:1'))
print('exit codes', pA.wait(), pB.wait(), 'hours', round((time.time() - t0) / 3600, 2))

In [ ]:
import json, glob
for p in sorted(glob.glob('/kaggle/working/tune/set_0*/selection.json')):
    S = json.load(open(p)); print(p)
    for k, v in S.items():
        print('  ', k, {kk: v[kk] for kk in v if kk in ('config', 'mean_val_ap', 'final_epochs', 'n_tied_within_0.001', 'hidden', 'feature_index', 'sign', 'n_features')})
for p in sorted(glob.glob('/kaggle/working/tune/log_*.txt')):
    print(open(p).read()[-3000:])